In [ ]:
!pip install -q langchain langchain-google-genai google-generativeai tiktoken langchain-community
!pip install langchain langchain-google-genai google-generativeai tiktoken faiss-cpu huggingface_hub langchain-community langgraph
!pip install -U langchain-groq
!pip install -U langchain-huggingface sentence-transformers
!pip install -U langchain-community faiss-cpu
!pip install tavily-python
!pip install pypdf
!pip install langchain-text-splitters
!pip install duckduckgo-search
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent
import uuid
import time

/tmp/ipykernel_16109/2226916171.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
#LLM
groq_api_key = "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"
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=groq_api_key,
    temperature=0,
    max_tokens=500,
    max_retries=3,  # retry automatically
    request_timeout=60
)

In [ ]:
#Searching tool
search_tool = DuckDuckGoSearchResults()

#Vectorstore (will set in main)
vectorstore = None

In [ ]:
def load_pdfs(pdf_paths):
  all_text = ""
  for path in pdf_paths:
    loader = PyPDFLoader(path)
    documents = loader.load()
    for doc in documents:
        all_text += doc.page_content + "\n"
    print(f"Loaded: {path}")
  return all_text


def chunk_text(text, chunk_size=500):
  chunks = []
  for i in range(0, len(text), chunk_size):
    chunk = text[i:i+chunk_size]
    if len(chunk) > 200:
      chunks.append(chunk)
  return chunks or [text]

# Load embeddings once at the top level, not inside the function
print("Loading embeddings model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embeddings model loaded!")

def create_vector_store(texts):
  # Convert raw text chunks into LangChain Document objects
    documents = [
        Document(
            page_content=text,
            metadata={"id": str(uuid.uuid4())}
        ) for text in texts
    ]

    # Use the already loaded embeddings instead of reloading
    vector_store = FAISS.from_documents(documents, embeddings)

    return vector_store

Loading embeddings model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model loaded!


In [ ]:
@tool
def search_documents(query: str) -> str:
  """Search for relevant information from the loaded PDF documents."""

  #Check if vector store has been created
  if vectorstore is None:
    return "No Vector store not created."

  #Search for similar chunks in vector store
  docs_with_scores = vectorstore.similarity_search_with_score(query, k=3)

  # Filter out chunks that are not relevant enough
  # Lower score = more similar, so we keep score below 1.5
  relevant_docs = [doc for doc, score in docs_with_scores if score <1.5]

  if not relevant_docs:
    return "No relevant docs found."

  #combine relevant chunks into a single string
  combined = "\n".join([f"- {doc.page_content}" for doc in relevant_docs])
  return f"Source: PDF Document\n\nRelevant documents:\n{combined}"


@tool
def dosage_calculator(weight_and_dose: str) -> str:
  """ Calculate total dosage given patient weight and dose per kg.
  Input format: 'wieght_kg, dose_per_kg, e.g. '70, 5'
  """
  try:
    #split the input string into weight and dose
    parts = weight_and_dose.split(",")
    weight = float(parts[0].strip())
    dose_per_kg = float(parts[1].strip())

    #Calculate tottal dosa
    total_dose = weight * dose_per_kg

    return f"Source: Dosage Calculator\n\nTotal dosage: {total_dose}mg (Weight: {weight}kg × Dose: {dose_per_kg}mg/kg)"
  except Exception as e:
    return f"Error calculating dosage: {str(e)}"

@tool
def search_drug_interactions(drugs: str) -> str:
    """Search for interactions between specified drugs.
    Input format: 'drug1, drug2' e.g. 'aspirin, ibuprofen'
    """
    try:
        # DuckDuckGo returns a single string directly
        results = search_tool.invoke({"query": f"drug interactions between {drugs}"})
        return f"Source: Internet Search\n\n{results}"
    except Exception as e:
        return f"Error searching drug interactions: {str(e)}"

In [ ]:
system_prompt = """You are a medical research assistant. You ONLY have access to these EXACT tools:
1. search_documents
2. dosage_calculator
3. search_drug_interactions

STRICT RULES:
1. NEVER call any tool not listed above
2. NEVER use brave_search, google_search or any other search tool
3. Use ONE tool only once per question
4. After getting the tool result, give your final answer immediately
5. Do NOT keep reasoning after you have an answer
6. ALWAYS format your final answer EXACTLY like this:

Answer: [your answer here]
Source: [where the information came from]

Never skip the Source line. Never skip the Answer line."""

def main():
    global vectorstore

    # --- Load and index PDF documents FIRST ---
    try:
        pdf_paths = ["/content/Introduction-to-research-and-ethics-Iveta-6-Feb.pdf"]
        pdf_text = load_pdfs(pdf_paths)
        chunks = chunk_text(pdf_text)
        chunks = chunks[:20]
        vectorstore = create_vector_store(chunks)
        print("Vector store created successfully!")
    except Exception as e:
        print(f"Vector store creation failed: {str(e)}")
        vectorstore = None

    # --- Set up tools and agent AFTER vector store ---
    tools = [search_documents, dosage_calculator, search_drug_interactions]
    agent = create_react_agent(llm, tools, prompt=system_prompt)

    # --- Test questions ---
    questions = [
        "What should I do before I start research?",
        "What is the dosage for a 70kg patient at 5mg/kg?",
        "What are the interactions between aspirin and ibuprofen?"
    ]

    # --- Run each question through the agent ---
    for question in questions:
        print(f"\nQuestion: {question}")
        response = agent.invoke(
            {"messages": [("human", question)]},
            config={"recursion_limit": 15}
        )
        print(response['messages'][-1].content)
    print("-" * 50)
    time.sleep(2)

main()

Loaded: /content/Introduction-to-research-and-ethics-Iveta-6-Feb.pdf
Vector store created successfully!

Question: What should I do before I start research?


/tmp/ipykernel_16109/3984477568.py:36: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=system_prompt)


Answer: Before starting research, it is essential to have a clear idea of what medical research involves, including key ethical and governance issues, and to systematically collect and synthesise literature to support further research project design and planning. This includes developing a protocol, obtaining ethical approval, and ensuring informed consent from participants.
Source: PDF Document

Question: What is the dosage for a 70kg patient at 5mg/kg?
Answer: 350.0mg
Source: Dosage Calculator

Question: What are the interactions between aspirin and ibuprofen?
Answer: Taking aspirin and ibuprofen together can increase your risk of developing stomach ulcers or internal bleeding and may also block aspirin from working properly.
Source: Internet Search
--------------------------------------------------


In [ ]:
# Quick diagnostic - run this separately
results = search_tool.invoke({"query": "drug interactions between aspirin and ibuprofen"})
print(type(results))
print(results)

<class 'str'>
snippet: Drug-drug interactions: This is the most common type of drug interaction and involves one drug interacting with another. If you take many medicines, your chances for this type of interaction increases., title: Drug Interaction Checker - Find Unsafe Combinations, link: https://www.drugs.com/drug_interactions.html, snippet: Interactions described include those between aspirin and ibuprofen, aspirin and other nonsteroidal anti-inflammatory drugs (NSAIDs), and the thienopyridine, clopidogrel, and drugs inhibiting CYP2C19, notably the proton pump inhibitors (PPI) omeprazole and esomeprazole., title: Antiplatelet drug interactions - University of Dundee Discovery Portal, link: https://discovery.dundee.ac.uk/en/publications/antiplatelet-drug-interactions/, snippet: Laboratory studies and studies in normal volunteers have suggested a possible adverse interaction between aspirin and ibuprofen. It is thought that ibuprofen prevents aspirin from gaining access to platelet c